# DBSCAN

Density-based clustering on California housing features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

target = data_df['median_house_value'].to_numpy()
raw_lon = data_df['longitude'].to_numpy()
raw_lat = data_df['latitude'].to_numpy()
X_all = data_df.drop(columns=['median_house_value']).to_numpy()

scaler = StandardScaler()
X_all = scaler.fit_transform(X_all)

rng = np.random.default_rng(42)
sub = rng.choice(len(X_all), 5000, replace=False)
X = X_all[sub]
t = target[sub]
lon = raw_lon[sub]
lat = raw_lat[sub]

print(X.shape)

## Baseline

In [ ]:
db = DBSCAN(eps=1.5, min_samples=10, n_jobs=-1)
labels = db.fit_predict(X)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())
print(f"Clusters: {n_clusters}")
print(f"Noise points: {n_noise}")
print("Cluster sizes:", dict(zip(*np.unique(labels, return_counts=True))))

## k-distance plot to pick eps

In [ ]:
k = 10
nn = NearestNeighbors(n_neighbors=k).fit(X)
d, _ = nn.kneighbors(X)
kd = np.sort(d[:, -1])

plt.figure(figsize=(9, 5))
plt.plot(kd)
plt.xlabel('points sorted')
plt.ylabel(f'{k}-NN distance')
plt.grid(alpha=0.3)
plt.show()

## eps sweep

In [ ]:
eps_list = [0.5, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]
results = []

for e in eps_list:
    m = DBSCAN(eps=e, min_samples=10, n_jobs=-1).fit(X)
    nc = len(set(m.labels_)) - (1 if -1 in m.labels_ else 0)
    nn = int((m.labels_ == -1).sum())
    results.append((e, nc, nn))
    print(f"eps={e:<5} clusters={nc:<3} noise={nn}")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].plot([r[0] for r in results], [r[1] for r in results], 'o-')
axs[0].set_xlabel('eps')
axs[0].set_ylabel('# clusters')
axs[0].grid(alpha=0.3)
axs[1].plot([r[0] for r in results], [r[2] for r in results], 'o-', color='coral')
axs[1].set_xlabel('eps')
axs[1].set_ylabel('# noise points')
axs[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## min_samples sweep

In [ ]:
for ms in [3, 5, 10, 20, 50]:
    m = DBSCAN(eps=1.5, min_samples=ms, n_jobs=-1).fit(X)
    nc = len(set(m.labels_)) - (1 if -1 in m.labels_ else 0)
    nn = int((m.labels_ == -1).sum())
    print(f"min_samples={ms:<3} clusters={nc:<3} noise={nn}")

## PCA projection

In [ ]:
Z = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(10, 7))
uniq = np.unique(labels)
for c in uniq:
    mask = labels == c
    if c == -1:
        plt.scatter(Z[mask, 0], Z[mask, 1], s=6, c='black', alpha=0.3, label='noise')
    else:
        plt.scatter(Z[mask, 0], Z[mask, 1], s=6, alpha=0.6, label=f'C{c}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3, loc='best')
plt.grid(alpha=0.3)
plt.show()

## Geographic view

In [ ]:
plt.figure(figsize=(10, 8))
noise_mask = labels == -1
plt.scatter(lon[noise_mask], lat[noise_mask], s=6, c='black', alpha=0.3, label='noise')
plt.scatter(lon[~noise_mask], lat[~noise_mask], s=6,
            c=labels[~noise_mask], cmap='tab10', alpha=0.6)
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.legend(markerscale=3)
plt.grid(alpha=0.3)
plt.show()

## DBSCAN on geography only

In [ ]:
geo = StandardScaler().fit_transform(np.c_[lon, lat])
db_geo = DBSCAN(eps=0.15, min_samples=20, n_jobs=-1).fit(geo)
gl = db_geo.labels_
nc = len(set(gl)) - (1 if -1 in gl else 0)
print(f"Geo clusters: {nc}  noise: {int((gl == -1).sum())}")

plt.figure(figsize=(10, 8))
m = gl == -1
plt.scatter(lon[m], lat[m], s=6, c='black', alpha=0.3, label='noise')
plt.scatter(lon[~m], lat[~m], s=6, c=gl[~m], cmap='tab20', alpha=0.7)
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.legend(markerscale=3)
plt.grid(alpha=0.3)
plt.show()